1. Naloga
Pri reˇsevanju kolokvija boste potrebovali knjiˇznice socket, select in Crypto
Najprej naredite par public, private key.
Napiˇsite program streˇznika in uporabnika. Uporabnik naj streˇzniku vsako sekundo poˇslje
trenutni ˇcas. Uporabnik sporoˇcila enkriptira, streˇznik pa jih dekriptira ter izpiˇse.
Sporoˇcila naj bodo enkriptirana z asimetriˇcno enkripcijo.

In [ ]:
from Crypto.PublicKey import RSA

bits = 2048
key = RSA.generate(bits)

private_key = key.export_key()
with open("private.pem", "wb") as f:
    f.write(private_key)

public_key = key.publickey().export_key()
with open("public.pem", "wb") as f:
    f.write(public_key)

print("Ključi generirani")

In [ ]:
from Crypto.PublicKey import RSA
from Crypto.Cipher import PKCS1_OAEP
import socket

HOST = "127.0.0.1"
PORT = 5000

private_key = RSA.import_key(open("private.pem").read())
cipher_rsa = PKCS1_OAEP.new(private_key)

server = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
server.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
server.bind((HOST, PORT))
server.listen(1)

print(f"Server posluša na {HOST}:{PORT}")

conn, addr = server.accept()
print("Povezan:", addr)

while True:
    encrypted_message = conn.recv(private_key.size_in_bytes())

    if not encrypted_message:
        break

    decrypted = cipher_rsa.decrypt(encrypted_message)
    print("Prejeto:", decrypted.decode())

conn.close()
server.close()

In [ ]:
from Crypto.PublicKey import RSA
from Crypto.Cipher import PKCS1_OAEP
import socket
import time
from datetime import datetime

HOST = "127.0.0.1"
PORT = 5000

public_key = RSA.import_key(open("public.pem").read())
cipher_rsa = PKCS1_OAEP.new(public_key)

client = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
client.connect((HOST, PORT))

while True:
    current_time = datetime.now().strftime("%H:%M:%S")

    encrypted = cipher_rsa.encrypt(current_time.encode())

    client.sendall(encrypted)

    print("Poslano:", current_time)

    time.sleep(1)